---
title: Validating XML in Python
---

:::{warning} This section requires significant work to complete
Currently the notebook outlines structure, with runnable code blocks, and bullet prompts only.
Every bullet marked **TODO** requires drafting and may need some additional planning or research.
:::

## Two kinds of validation

TODO — bullets to expand into prose:

- *Structural* validation: does the document follow the rules the schema sets out —
  which elements, in what order, with which attributes? Answered mechanically, yes or no.
- *Content* validation: are the values inside those elements the right values? A record
  can pass the schema and still be wrong for local purposes. Handled with the tools from
  the previous sections (XPath, regular expressions) plus a local rule, not a schema.
- This is the point at which ElementTree runs out: ET has no validation API of any kind.
  `lxml` does — `XMLSchema`, `RelaxNG`, `Schematron`, and `DTD` — which is the reason
  the earlier sections introduced it. (TODO: this pays off the ET → lxml pivot.)
- References: [lxml validation documentation](https://lxml.de/validation.html);
  [EAD3 schema and tag library](https://www.loc.gov/ead/); TODO — add the local
  metadata-profile reference used in Part 3.

In [1]:
# Setup
from pathlib import Path
from lxml import etree
import re

DATA = Path('..', 'data')
OUT = Path('..', 'output')

# a finding aid that conforms to the EAD3 schema
EAD_valid = DATA / 'sample-ead-superior-full.xml'
# a finding aid that does not — the counterexample, used for the problem set below
EAD_invalid = DATA / 'sample-ead-superior.xml'
# a local copy of the EAD3 XSD, so validation does not depend on a network call
EAD3_XSD = DATA / 'ead3.xsd'

ns = {
    'ead' : 'http://ead3.archivists.org/schema/'
}

TODO — bullets on the setup:

- Note the local schema copy: validating against a URL works but fails offline and puts
  load on the standards body's server. Keeping the XSD alongside the data is the norm.
- TODO: say where `ead3.xsd` came from and how a reader would fetch their own.

## Validating structure against a schema

TODO — bullets to expand:

- The schema is compiled once and reused against many documents.
- The EAD3 sample points at a RelaxNG schema in its `xml-model` declaration
  (`ead3.rng`) but here we use the XSD. TODO — a sentence on why both exist and
  that `etree.RelaxNG()` works the same way if a reader prefers the RNG.

The code splits into four blocks: compile the schema, validate a record that
conforms, validate one that does not, then read the errors.

In [2]:
# Block 1 — compile the schema once, then reuse it
schema_doc = etree.parse(EAD3_XSD)
schema = etree.XMLSchema(schema_doc)

In [3]:
# Block 2 — the expected case: a record that conforms
tree = etree.parse(EAD_valid)

schema.validate(tree)

True

TODO — bullets on the two calling styles:

- `.validate()` returns a boolean and does not interrupt anything. Use it when you
  want to sort a batch of records into good and bad.
- `.assertValid()` raises `etree.DocumentInvalid` instead. Use it in a processing
  pipeline where an invalid record should stop the run rather than pass through.

In [4]:
# the same check, raising instead of returning
try:
    schema.assertValid(tree)
    print(f'{EAD_valid.name} is valid EAD3')
except etree.DocumentInvalid as err:
    print(f'{EAD_valid.name} is not valid:\n{err}')

sample-ead-superior-full.xml is valid EAD3


### A counterexample

TODO — bullets to expand:

- The second sample is well-formed XML: it parses without complaint. Well-formed and
  valid are different claims, and only the schema can settle the second one.
- TODO — describe where this record came from and why it looks plausible at a glance.

In [14]:
# Block 3 — the counterexample
bad_tree = etree.parse(EAD_invalid)

schema.validate(bad_tree)

False

In [15]:
# Block 4 — the error log is where the diagnosis happens
for error in schema.error_log:
    print(f'line {error.line}: {error.message}')

line 3: Element '{http://ead3.archivists.org/schema/}control': Missing child element(s). Expected is ( {http://ead3.archivists.org/schema/}maintenancestatus ).
line 23: Element '{http://ead3.archivists.org/schema/}title': Character content other than whitespace is not allowed because the content type is 'element-only'.
line 23: Element '{http://ead3.archivists.org/schema/}title': Missing child element(s). Expected is ( {http://ead3.archivists.org/schema/}part ).
line 27: Element '{http://ead3.archivists.org/schema/}bioghist': Character content other than whitespace is not allowed because the content type is 'element-only'.
line 27: Element '{http://ead3.archivists.org/schema/}bioghist': Missing child element(s). Expected is one of ( {http://ead3.archivists.org/schema/}head, {http://ead3.archivists.org/schema/}chronlist, {http://ead3.archivists.org/schema/}list, {http://ead3.archivists.org/schema/}table, {http://ead3.archivists.org/schema/}p, {http://ead3.archivists.org/schema/}blockquo

TODO — bullets on reading the log:

- Each entry is an error object, not a string: `.line`, `.column`, `.message`,
  `.type_name`, `.domain_name`, `.level_name`, `.filename`. TODO — show one
  formatted report that a cataloger could act on.
- `error_log` is replaced on every `.validate()` call, so read it before validating
  the next document.
- The line number points at the element that *broke* the rule, which is not always the
  element a person would think to edit. TODO — this is worth a sentence; it is the most
  common source of confusion.

TODO — the eight errors in this file fall into three groups, which is a convenient
teaching structure:

1. *Missing required child* — `control` has no `maintenancestatus`.
2. *Element-only content type* — `title`, `bioghist`, and `c01` hold bare text where
   the schema expects child elements (`part`, `p`, `did` …). TODO: this is the EAD3
   "everything is wrapped" habit, and the error message does not say so plainly.
3. *Wrong sequence* — a `c02` appears where a `c01` is expected.

### Problem set: diagnose and correct

:::{exercise} TODO — write the prompt
Students receive the invalid record and the schema, and work through the log until
the record validates.

TODO — bullets toward the prompt:

- How many errors does the schema report, and how many distinct *problems* is that?
  (Not the same number — several messages describe one defect.)
- Fix the missing `maintenancestatus` first, revalidate, and notice that the error
  count changes. Why does fixing one thing surface or clear others?
- Two of the errors are the same mistake in three places. Which?
- Correct the record so it validates, then save it (next block).
:::

:::{admonition} Drafting notes — other samples that could serve
:class: tip, dropdown
- `data/bhl-ead-toy-papers.xml` fails against `ead3.xsd` with a single error —
  "No matching global declaration available for the validation root" — because it is
  EAD 2002, DTD-based, and carries no namespace. That is a *different and better*
  question ("valid against *what*?"), so it could be an extension problem rather than
  a replacement. It cannot be fixed by editing; it needs the right schema.
- TODO — decide whether to author a third sample with deliberately seeded errors of a
  chosen type (bad attribute value, out-of-vocabulary term) rather than the incidental
  ones in `sample-ead-superior.xml`.
:::

In [ ]:
# TODO — repair scaffold for the problem set
# Students edit the tree, then re-run validation until it passes.

# ... corrections here ...

schema.validate(bad_tree)

## Validating content for local conformance

The above techniques are useful for validating the XML
structure (_well-formedness_) and whether or not the data follows the schema rules (_valid_).
Beyond these, however, most cultural heritage collections have local rules
that serve the needs of their collections. For example, perhaps a local history collection uses a local name authority list to augment the Library of Congress's name authority.
Local rules can take different shapes and respond to different needs. The example below assumes that the identifiers used in records must conform to a specific format or vocabulary.
This would not be specified in an XSD, so these kinds of errors require a different validation approach. The same basic sequence to check, validate, and repair
remains intact.

TODO — bullets to expand:

- The record that passed the schema still has an identifier problem. A schema constrains
  structure; it cannot know that this repository writes accession numbers one particular way.
- Local rules like this live in a metadata application profile, not in the XSD. TODO —
  connect forward to Part 3 / the MAP assignment.
- This is where regular expressions come back: the rule is a pattern, and checking it is
  a loop over the elements XPath returns.

Two blocks: look at what is actually in the records, then express the rule and test it.

In [13]:
# Block 1 — what identifiers does this finding aid actually carry?
tree = etree.parse(EAD_valid)

for unitid in tree.iterfind('.//ead:unitid', namespaces=ns):
    print(f'{unitid.text!r:40} {dict(unitid.attrib)}')

'00.1332'                                {}
'/repositories/2/resources/1'            {'localtype': 'aspace_uri'}
'/repositories/2/archival_objects/3'     {'localtype': 'aspace_uri'}
'00-1332-01'                             {}
'/repositories/2/archival_objects/4'     {'localtype': 'aspace_uri'}
'00-1332-02'                             {}
'/repositories/2/archival_objects/1'     {'localtype': 'aspace_uri'}
'deep-six'                               {}
'/repositories/2/archival_objects/2'     {'localtype': 'aspace_uri'}
'deep-six'                               {}


TODO — bullets on what the output shows:

- Two populations of identifier in one element name, told apart by `@localtype`:
  ArchivesSpace URIs (`/repositories/2/resources/1`) and local accession numbers.
- The local numbers are not consistent: `00.1332` uses a period where the others use a
  hyphen, and `deep-six` appears twice and is not an identifier at all.
- TODO — this is the argument for content validation in one example: the file is valid
  EAD3 and the identifiers are still unusable.

In [ ]:
# Block 2 — express the local rule as a pattern, then test every identifier against it
# local accession numbers should look like 00-1332, optionally with a -01 item suffix
unitid_pattern = re.compile(r'^\d{2}-\d{4}(-\d{2})?$')

for unitid in tree.iterfind('.//ead:unitid', namespaces=ns):
    if unitid.get('localtype') == 'aspace_uri':
        continue                      # a different identifier system, not ours to check
    value = unitid.text or ''
    if unitid_pattern.match(value):
        print(f'ok      {value}')
    else:
        print(f'CHECK   {value!r}')

TODO — bullets to expand:

- Reading the pattern out loud, group by group. TODO — keep this short; the regex
  intro belongs to the earlier section.
- The `continue` is doing real work: without it every ArchivesSpace URI is reported as
  a failure. Deciding *which* elements a rule applies to is half of content validation.
- TODO — a note that `unitid.text` can be `None` for an empty element, hence `or ''`.

:::{admonition} Drafting notes — moved from the MODS notebook
:class: tip, dropdown
This block replaces Activity 7 in `xml-working-with-MODS.ipynb`, which stood as
`#### Data validation - ADD new sample file TODO` in both the 2024 and 2025 course
versions. Retargeted from MODS `identifier` / LCWA call numbers to EAD `unitid`,
so the whole section works from one pair of files.

Reusable as written from the MODS notebook: the `re.compile()` + loop + branch shape,
and the `f'invalid LCCN: {...}'` reporting line. The MODS pattern
`r'[a-z]{4}N\d{7}'` and its `re.match` usage carry over directly if the MODS records
stay in play as a second example.
:::

## From validation to repair

TODO — bullets to expand:

- Validation that only reports is half a workflow. Once a defect is identified by rule,
  the same rule usually says what the corrected value should be.
- The sequence is: find by XPath → test against the rule → set or correct → revalidate →
  write. TODO — name this arc explicitly; it recurs in Part 3 with JSON Schema.
- Revalidating after the repair is the step people skip. TODO — make the point that a
  repair can introduce a new schema error.

:::{admonition} Drafting notes — reusable material for this arc
:class: tip, dropdown
**Code that transfers with little change** (from `xml-working-with-MODS.ipynb`,
2025 course version):

- Cell 81, "lxml insert attributes to make a more complete metadata record" — the
  `re.match` → `identifier.attrib[...] = ...` loop. Already lxml, already the right
  shape; swap `mods:identifier` for `ead:unitid`.
- Cell 87, "lxml write file" — `register_namespace()` calls plus
  `metadata.write(newfile, xml_declaration=True, encoding='utf-8', method='xml')`.
  This is the closing block of the arc and needs only the new filename.
- The lab-8 key (`si676-2025-activities/_posts/2025-11-06-lab-8-key.md`, line ~144)
  has a better reporting line than the notebook:
  `print('full XML element after validation and attribute addition:\n',
  etree.tostring(identifier).decode())` — shows the before/after of the repair.

**Prose that transfers**:

- `xml-working-with-MODS.ipynb` cell 75, "Data Addition or Modification" — the framing
  paragraph with the sample `<identifier>` block showing what is missing. The argument
  is written; only the element names change.
- `xml-intro-basic-functions-ET.ipynb` cell 31, "Writing XML with etree" — the
  namespace-registration explanation, including the MODS User Guide quotation on when
  to use prefixes. Reusable verbatim for the write-out block.
- The MODS notebook's opening paragraph already claims lxml "provides support for XML
  validation" without demonstrating it. TODO — that sentence can now point here.

**Not reusable, must be written**: everything in the schema-validation section above.
No `etree.XMLSchema`, `assertValid`, `RelaxNG`, `Schematron`, or DTD call appears
anywhere in the 2024 or 2025 course materials.
:::

In [ ]:
# TODO — repair block
# find by XPath, test against the rule, correct, report the change

# ... corrections here ...

In [ ]:
# TODO — revalidate before writing
schema.validate(tree)

In [ ]:
# TODO — write the corrected record
# newfile = OUT / 'sample-ead-superior-corrected.xml'
# tree.write(newfile, xml_declaration=True, encoding='utf-8', method='xml')

## Summary

TODO — bullets:

- Well-formed, schema-valid, and locally correct are three different claims.
- `lxml` answers the second mechanically; the third needs a rule you write.
- TODO — hand off to the next section.